# Notebook 06 — Rating Distribution

## 1. Setup

In [1]:
import sys
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
from IPython.display import display, HTML

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

INTERIM_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Environment ready.")

Environment ready.


## 2. Load data

In [2]:
df     = pd.read_parquet(INTERIM_DIR / "analysis_ready.parquet")
sbux   = df[df["brand_category"] == "Starbucks"]
market = df[df["brand_category"] != "Starbucks"]

print(f"Starbucks reviews : {len(sbux):,}")
print(f"Market reviews    : {len(market):,}")
print(f"\nStarbucks avg rating : {sbux['review_stars'].mean():.2f}")
print(f"Market avg rating    : {market['review_stars'].mean():.2f}")
print(f"\nStarbucks star rating counts:")
display(sbux["review_stars"].value_counts().sort_index().to_frame("count"))

Starbucks reviews : 11,675
Market reviews    : 370,324

Starbucks avg rating : 2.90
Market avg rating    : 3.97

Starbucks star rating counts:


,count
review_stars,
1.0,3937
2.0,1620
3.0,1171
4.0,1562
5.0,3385


## 3. Star rating distribution

In [3]:
star_counts = sbux["review_stars"].value_counts().sort_index()
star_pcts   = (star_counts / len(sbux) * 100).round(1)

star_df = pd.DataFrame({"count": star_counts, "pct": star_pcts})
star_df.index = [f"{int(s)} star" for s in star_df.index]
display(star_df)

fig = go.Figure(go.Pie(
    labels=[f"{int(s)}\u2605" for s in star_counts.index],
    values=star_counts.values,
    marker=dict(colors=["#d62728", "#ff7f0e", "#bcbd22", "#2ca02c", "#1f77b4"]),
    textinfo="label+percent",
    hole=0.3,
    sort=False,
))
fig.update_layout(
    title=dict(text="Starbucks — Star Rating Distribution", font=dict(size=16)),
    paper_bgcolor="white",
    width=600, height=480,
    margin=dict(t=60, b=20, l=20, r=20),
)
fig.write_html(str(FIGURES_DIR / "06_star_distribution.html"))
fig.show()

,count,pct
1 star,3937,33.7
2 star,1620,13.9
3 star,1171,10.0
4 star,1562,13.4
5 star,3385,29.0


## 4. Satisfaction tier breakdown

In [4]:
tier_order  = ["Critical", "Neutral", "Positive"]
tier_colors = {"Critical": "#d62728", "Neutral": "#ff7f0e", "Positive": "#2ca02c"}
tier_counts = sbux["star_tier"].value_counts()

tier_df = pd.DataFrame({
    "Tier": tier_order,
    "Count": [tier_counts.get(t, 0) for t in tier_order],
    "Pct": [round(tier_counts.get(t, 0) / len(sbux) * 100, 1) for t in tier_order],
})
display(tier_df)

fig = go.Figure(go.Pie(
    labels=tier_order,
    values=[tier_counts.get(t, 0) for t in tier_order],
    marker=dict(colors=[tier_colors[t] for t in tier_order]),
    textinfo="label+percent",
    hole=0.3,
))
fig.update_layout(
    title=dict(text="Starbucks — Satisfaction Tier Breakdown", font=dict(size=16)),
    paper_bgcolor="white",
    width=600, height=480,
    margin=dict(t=60, b=20, l=20, r=20),
)
fig.write_html(str(FIGURES_DIR / "06_star_tier.html"))
fig.show()

,Tier,Count,Pct
0,Critical,5557,47.6
1,Neutral,1171,10.0
2,Positive,4947,42.4


## 5. Rating trend (Starbucks vs. market)

In [5]:
sbux_avg   = sbux.groupby("year")["review_stars"].mean().round(2)
market_avg = market.groupby("year")["review_stars"].mean().round(2)

rating_trend = pd.DataFrame({
    "Starbucks": sbux_avg,
    "Market": market_avg,
    "Gap": (market_avg - sbux_avg).round(2),
})
display(rating_trend)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=sbux_avg.index.astype(str), y=sbux_avg.values,
    mode="lines+markers", name="Starbucks",
    line=dict(color="#00704A", width=2.5),
    marker=dict(size=8),
))
fig.add_trace(go.Scatter(
    x=market_avg.index.astype(str), y=market_avg.values,
    mode="lines+markers", name="Market avg",
    line=dict(color="#aaaaaa", width=2, dash="dash"),
    marker=dict(size=7),
))
fig.update_layout(
    title=dict(text="Average Star Rating by Year — Starbucks vs. Market", font=dict(size=16)),
    xaxis_title="Year", yaxis_title="Avg Star Rating",
    plot_bgcolor="white", paper_bgcolor="white",
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=True, gridcolor="#eeeeee", range=[1, 5]),
    legend=dict(x=0.75, y=0.15),
    width=820, height=460,
    margin=dict(t=60, b=50, l=60, r=40),
)
fig.write_html(str(FIGURES_DIR / "06_rating_trend.html"))
fig.show()

,Starbucks,Market,Gap
year,,,
2017,2.99,3.94,0.95
2018,2.93,3.96,1.03
2019,2.89,3.96,1.07
2020,2.88,4.04,1.16
2021,2.76,3.97,1.21


## 6. Sentiment score trend

In [6]:

# Sentiment score trend over time — Starbucks vs market
sbux_sent   = sbux.groupby("year")["sentiment_score"].mean().round(3)
market_sent = market.groupby("year")["sentiment_score"].mean().round(3)

display(pd.DataFrame({"Starbucks": sbux_sent, "Market": market_sent}))

fig_s = go.Figure()
fig_s.add_trace(go.Scatter(
    x=sbux_sent.index.astype(str), y=sbux_sent.values,
    mode="lines+markers", name="Starbucks",
    line=dict(color="#00704A", width=2.5), marker=dict(size=8),
))
fig_s.add_trace(go.Scatter(
    x=market_sent.index.astype(str), y=market_sent.values,
    mode="lines+markers", name="Market",
    line=dict(color="#aaaaaa", width=2, dash="dot"), marker=dict(size=6),
))
fig_s.update_layout(
    title=dict(text="Starbucks vs Market — Avg VADER Sentiment Score (2017–2021)", font=dict(size=16)),
    xaxis_title="Year",
    yaxis_title="Avg Sentiment Score",
    plot_bgcolor="white", paper_bgcolor="white",
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=True, gridcolor="#eeeeee"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    width=720, height=420,
    margin=dict(t=70, b=50, l=60, r=40),
)
fig_s.write_html(str(FIGURES_DIR / "06_sentiment_trend.html"))
fig_s.show()


,Starbucks,Market
year,,
2017,0.301,0.701
2018,0.308,0.696
2019,0.277,0.691
2020,0.264,0.696
2021,0.207,0.671


## 7. Star-sentiment mismatch analysis

In [7]:
tbl = sbux.groupby("review_stars").agg(
    count=("review_id", "count"),
    avg_sentiment=("sentiment_score", "mean"),
).reset_index()
tbl["pct"]           = (tbl["count"] / len(sbux) * 100).round(1)
tbl["avg_sentiment"] = tbl["avg_sentiment"].round(3)
tbl["review_stars"]  = tbl["review_stars"].astype(int)
tbl.columns = ["Stars", "Count", "Avg Sentiment Score", "% of Reviews"]
tbl = tbl[["Stars", "Count", "% of Reviews", "Avg Sentiment Score"]]

print("Star rating vs. avg VADER sentiment score (Starbucks):")
display(tbl)

high_star_neg = ((sbux["review_stars"] >= 4) & (sbux["sentiment_label"] == "Negative")).sum()
low_star_pos  = ((sbux["review_stars"] <= 3) & (sbux["sentiment_label"] == "Positive")).sum()

print(f"\nStar-sentiment mismatch:")
print(f"  Low star (1-3)  + Positive sentiment : {low_star_pos:,}  ({low_star_pos/len(sbux)*100:.1f}%)")
print(f"  High star (4-5) + Negative sentiment : {high_star_neg:,}  ({high_star_neg/len(sbux)*100:.1f}%)")

Star rating vs. avg VADER sentiment score (Starbucks):


,Stars,Count,% of Reviews,Avg Sentiment Score
0,1,3937,33.7,-0.290
1,2,1620,13.9,-0.019
2,3,1171,10.0,0.333
3,4,1562,13.4,0.726
4,5,3385,29.0,0.851



Star-sentiment mismatch:
  Low star (1-3)  + Positive sentiment : 2,633  (22.6%)
  High star (4-5) + Negative sentiment : 163  (1.4%)


## Key Findings

- Ratings are polarized. 1-star (3,937, 33.7%) and 5-star (3,385, 29.0%) make up most reviews. The Critical tier (1-2 star) is 47.6% vs. 42.4% Positive (4-5 star).
- Starbucks averages 2.90 stars against a 3.97 market average.
- That gap is growing: 0.95 in 2017, 1.21 by 2021.
- VADER sentiment dropped from 0.301 (2017) to 0.207 (2021). Market stayed flat around 0.69.
- 22.6% of low-star (1-3) reviews contain positive sentiment language. These are mixed-experience customers who liked some things but left dissatisfied overall.